In [1]:
import os
path="/Users/gaiaandreoletti/Downloads/"
os.chdir(path)


In [2]:
from __future__ import annotations

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from scipy import stats


In [8]:
# -----------------------------
# 0) Config
# -----------------------------
DATA_DIR = "data"
META_PATH = os.path.join(DATA_DIR, "metadata.csv")
RNA_PATH  = os.path.join(DATA_DIR, "transcriptomics.csv")
PROT_PATH = os.path.join(DATA_DIR, "proteomics.csv")

OUT_DIR = "outputs"
os.makedirs(OUT_DIR, exist_ok=True)

GENE_OF_INTEREST = "GENE001"
PROT_OF_INTEREST = "PROT001"

BASELINE_LABEL = "baseline"
WEEK4_LABEL = "week4"

np.random.seed(0)  # for jitter reproducibility

def save_text(s: str, path: str):
    with open(path, "w") as f:
        f.write(s)

print(">>> Step 0: Config")
print("META_PATH:", META_PATH)
print("RNA_PATH :", RNA_PATH)
print("PROT_PATH:", PROT_PATH)
print("OUT_DIR  :", OUT_DIR)
print()


## load files
#meta = pd.read_csv("data/metadata.csv")
#rna  = pd.read_csv("data/transcriptomics.csv")
#prot = pd.read_csv("data/proteomics.csv")

meta = load_and_clean_metadata(META_PATH)
rna_raw = load_matrix_flexible(RNA_PATH)
prot_raw = load_matrix_flexible(PROT_PATH)

print("Files loaded successfully")

print("Metadata shape:", meta.shape)
print("\nMetadata columns:")
print(meta.columns.tolist())

print("\nFirst 5 rows:")
print(meta.head())


print("RNA raw shape:", rna.shape)
print("\nRNA columns (first 10):")
print(rna.columns[:10])
print("\nRNA rows (first 5):")
print(rna.iloc[:5, :5])

## check if GEN001 exists
print("GENE001 present:", "GENE001" in rna.iloc[:, 0].values)


print("Proteomics raw shape:", prot.shape)
print("\nProteomics columns (first 10):")
print(prot.columns[:10])
print("\nProteomics rows (first 5):")
print(prot.iloc[:5, :5])
print("PROT001 present:", "PROT001" in prot.iloc[:, 0].values)



>>> Step 0: Config
META_PATH: data/metadata.csv
RNA_PATH : data/transcriptomics.csv
PROT_PATH: data/proteomics.csv
OUT_DIR  : outputs

Files loaded successfully
Metadata shape: (40, 7)

Metadata columns:
['sample_id', 'subject_id', 'treatment', 'timepoint', 'batch', 'site', 'dose']

First 5 rows:
       sample_id subject_id treatment timepoint batch   site  dose
0  S001_baseline       S001   placebo  baseline    B2  SiteA     0
1     S001_week4       S001   placebo     week4    B1  SiteB     0
2  S002_baseline       S002       ASO  baseline    B1  SiteA    50
3     S002_week4       S002       ASO     week4    B1  SiteB    25
4  S003_baseline       S003   placebo  baseline    B1  SiteA     0
RNA raw shape: (250, 41)

RNA columns (first 10):
Index(['gene', 'S001_baseline', 'S001_week4', 'S002_baseline', 'S002_week4',
       'S003_baseline', 'S003_week4', 'S004_baseline', 'S004_week4',
       'S005_baseline'],
      dtype='object')

RNA rows (first 5):
      gene  S001_baseline  S001_week